In [1]:
import pandas as pd

bureau = pd.read_csv("../data/bureau.csv")
print(bureau.shape)
bureau.head()

(1716428, 17)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [6]:
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_COUNT=("SK_ID_BUREAU", "count"),
    BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
    BUREAU_CREDIT_SUM=("AMT_CREDIT_SUM", "sum"),
    BUREAU_CREDIT_SUM_DEBT=("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_CREDIT_SUM_OVERDUE=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    BUREAU_CNT_CREDIT_PROLONG=("CNT_CREDIT_PROLONG", "sum"),
).reset_index()

# Derived ratio: how much of total historical credit is still owed
bureau_agg["BUREAU_DEBT_CREDIT_RATIO"] = (
    bureau_agg["BUREAU_CREDIT_SUM_DEBT"] / bureau_agg["BUREAU_CREDIT_SUM"]
)
import numpy as np
bureau_agg["BUREAU_DEBT_CREDIT_RATIO"] = bureau_agg["BUREAU_DEBT_CREDIT_RATIO"].replace(
    [np.inf, -np.inf], np.nan
)
print(bureau_agg.shape)
bureau_agg.head()

(305811, 8)


,SK_ID_CURR,BUREAU_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_CREDIT_SUM,BUREAU_CREDIT_SUM_DEBT,BUREAU_CREDIT_SUM_OVERDUE,BUREAU_CNT_CREDIT_PROLONG,BUREAU_DEBT_CREDIT_RATIO
0,100001,7,-735.000000,1453365.000,596686.5,0.0,0,0.410555
1,100002,8,-874.000000,865055.565,245781.0,0.0,0,0.284122
2,100003,4,-1400.750000,1017400.500,0.0,0.0,0,0.000000
3,100004,2,-867.000000,189037.800,0.0,0.0,0,0.000000
4,100005,3,-190.666667,657126.000,568408.5,0.0,0,0.864992


In [7]:
df = pd.read_csv("../data/application_train_features.csv")
df = df.merge(bureau_agg, on="SK_ID_CURR", how="left")

print(f"Shape before merge: previous dataset")
print(f"Shape after merge: {df.shape}")

# Applicants with no bureau history at all will have NaN here — meaningful signal, keep it
df["HAS_BUREAU_HISTORY"] = df["BUREAU_COUNT"].notnull().astype(int)

Shape before merge: previous dataset
Shape after merge: (307511, 138)


/var/folders/3d/_njm_nr56yj72jj4qzhn0ctc0000gp/T/ipykernel_29292/3781376263.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["HAS_BUREAU_HISTORY"] = df["BUREAU_COUNT"].notnull().astype(int)


In [8]:
df.to_csv("../data/application_train_bureau.csv", index=False)
print("Saved:", df.shape)


Saved: (307511, 139)


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay
X = df.drop(columns="TARGET")
y = df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numerical_features = X_train.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor_v2 = ColumnTransformer(transformers=[
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor_v2.fit(X_train)
X_train_processed = preprocessor_v2.transform(X_train)
X_test_processed = preprocessor_v2.transform(X_test)

# Reuse your Optuna-tuned hyperparameters — just retrain on richer features
lgbm_v2 = LGBMClassifier(
    n_estimators=349, learning_rate=0.026893915216046605, num_leaves=121,
    max_depth=10, min_child_samples=78, subsample=0.6576670035497199,
    colsample_bytree=0.6353716160445806,
    class_weight="balanced", random_state=42, verbose=-1
)
lgbm_v2.fit(X_train_processed, y_train)

y_pred_v2 = lgbm_v2.predict(X_test_processed)
y_prob_v2 = lgbm_v2.predict_proba(X_test_processed)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_v2))
print("Precision:", precision_score(y_test, y_pred_v2))
print("Recall   :", recall_score(y_test, y_pred_v2))
print("F1 Score :", f1_score(y_test, y_pred_v2))
print("ROC AUC  :", roc_auc_score(y_test, y_prob_v2))

/var/folders/3d/_njm_nr56yj72jj4qzhn0ctc0000gp/T/ipykernel_29292/3050117427.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()
/Users/swarnimsingh/Library/Python/3.11/lib/python/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/swarnimsingh/Library/Python/3.11/lib/python/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but 

Accuracy : 0.7456059054029884
Precision: 0.18431163917952356
Recall   : 0.6279959718026183
F1 Score : 0.28498309112512565
ROC AUC  : 0.7653843450547407


In [10]:
import joblib, shap

joblib.dump(lgbm_v2, "../app/lightgbm_model.pkl")
joblib.dump(preprocessor_v2, "../app/preprocessor.pkl")

explainer_v2 = shap.TreeExplainer(lgbm_v2)
joblib.dump(explainer_v2, "../app/shap_explainer.pkl")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['../app/shap_explainer.pkl']

### Bureau Data Integration — Results

Added aggregated features from `bureau.csv`. Results show a genuine tradeoff, 
not a clean win:

| Metric | LightGBM (deployed) | LightGBM + Bureau |
|--------|----------------------|--------------------|
| ROC-AUC | 0.762 | 0.765 |
| F1 | 0.277 | 0.285 |
| Precision | 0.175 | 0.184 |
| **Recall** | **0.662** | **0.628** |

**Conclusion:** Bureau history improved overall ranking ability (ROC-AUC) 
and precision, but reduced recall — the metric most important for this 
problem, since missing an actual defaulter is costlier than a false alarm 
on a safe applicant. Given this tradeoff, the bureau-enhanced model was 
**not** promoted to production; the currently deployed model (higher 
recall) was kept. This experiment is documented as a candidate for future 
work — likely combined with threshold tuning to recover recall while 
keeping the bureau signal, rather than adopted outright.